# Natural Language Processing — Lab 03
## Regex, Tokenization & Sentence Segmentation

| | |
|---|---|
| **Student** | Eman Amjad |
| **Roll Number** | FA23-BAI-013 |
| **Course** | Natural Language Processing |

This lab uses regular expressions to pull emails, order numbers, phone numbers, dates, percentages and money amounts out of raw text, then compares NLTK and spaCy for word tokenization, token-attribute filtering and sentence segmentation.

In [1]:
# Setup  |  Eman Amjad  |  FA23-BAI-013
import re
import difflib
from pathlib import Path

import pandas as pd
import nltk
import spacy
from nltk.tokenize import word_tokenize, sent_tokenize

# word_tokenize / sent_tokenize need the Punkt data; without it they raise a LookupError
for package in ["punkt", "punkt_tab"]:
    nltk.download(package, quiet=True)

# Load the trained English pipeline once and reuse it in every activity
nlp = spacy.load("en_core_web_sm")

print("NLTK version :", nltk.__version__)
print("spaCy version:", spacy.__version__)
print("Pipeline     :", nlp.meta["name"], nlp.meta["version"])
print("Components   :", nlp.pipe_names)

NLTK version : 3.10.4
spaCy version: 3.8.7
Pipeline     : core_web_sm 3.8.0
Components   : ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [2]:
# Data files for the home activities  |  Eman Amjad  |  FA23-BAI-013
# The manual refers to a "given" text block, document and paragraph without supplying them,
# so they are written here once and read back from disk in the home activities.
DATA_DIR = Path("data/lab03")
DATA_DIR.mkdir(parents=True, exist_ok=True)

support_tickets = """[Ticket #4471 | 2024-03-12 09:14]
From: sara.khan+orders@mail.shopnow.co.uk
Hi team, my order ORD-58392 arrived with only 3 of the 5 mugs I paid for ($48.50 total).
Please refund $19.40 or resend two mugs. You can call me at (415) 555-0198 after 6 pm.

[Ticket #4472 | 2024-03-12 10:02]
From: m.ali@techmart.pk
Orders ORD-61007 and ORD-61008 were both charged twice: 2 x $1,299.00 for the laptops.
I expect a refund of $2,598 within 7 days. Reach me at 212.555.7731 or billing@techmart.pk.

[Ticket #4473 | 2024-03-13 16:45]
From: j_doe99@gmail.com
Hello, I ordered twelve notebooks (ORD-70215) for €36 but received ten.
My backup number is 646 555 0142. Total spent this year is about £1,250, roughly 1500 dollars.
"""

product_review = (
    "I've been using the X200 headphones for 3 weeks now, and honestly? They're great... mostly. "
    "The battery doesn't last the advertised 30 hrs; I'm getting ~22 hrs at 60% volume. "
    "Customer support (support@soundwave.com) said they'd \"look into it\" -- but it's been 5 days. "
    "Price: $129.99 (down from $149.99). Would I recommend 'em? Yes, if you can't find the "
    "Sony WH-1000XM4 on sale. P.S. The U.S. version ships with a USB-C cable, not a 3.5mm one! "
    "Rating: 4/5, see https://soundwave.com/x200-review."
)

news_paragraph = (
    "Last year, Dr. Ayesha Malik joined the U.S. office of DataCore Inc. "
    "She now leads a team of 12 engineers at a startup valued at $1.2 billion. "
    "Her first project, an NLP tool for customer e-mails, cut response times by 35 per cent. "
    "\"We didn't expect results this fast,\" said J. R. Smith, the company's CTO. "
    "The tool now handles approx. 4,000 messages a day. "
    "Next quarter, the team plans to expand it to the U.K. and Canada."
)

files = {
    "support_tickets.txt": support_tickets,
    "product_review.txt": product_review,
    "news_paragraph.txt": news_paragraph,
}
for name, text in files.items():
    (DATA_DIR / name).write_text(text, encoding="utf-8")
    print(f"wrote {(DATA_DIR / name).as_posix()}  ({len(text)} characters)")

wrote data/lab03/support_tickets.txt  (716 characters)
wrote data/lab03/product_review.txt  (495 characters)
wrote data/lab03/news_paragraph.txt  (421 characters)


---
# In-Lab Activities

**Activity 1** — Demonstrate regex fundamentals (groups, quantifiers and character classes) using a sample sentence of my choice.

In [3]:
# Activity 1  |  Eman Amjad  |  FA23-BAI-013
sample_sentence = "The price of the item is $45.99, and my contact email is user2024@example.com."

# Character classes: \d = a digit, \w = a word character [a-zA-Z0-9_]
# Quantifiers: + = one or more, {2} = exactly two, {5,} = five or more
# Groups: (...) capture parts of the match so they can be read back separately
email_pattern = r"(\w+)@(\w+\.\w+)"
price_pattern = r"\$(\d+\.\d{2})"

email_match = re.search(email_pattern, sample_sentence)
if email_match:
    print("Full email matched:", email_match.group(0))
    print("Username group    :", email_match.group(1))
    print("Domain group      :", email_match.group(2))

price_match = re.search(price_pattern, sample_sentence)
if price_match:
    print("Price matched     :", price_match.group(1))

Full email matched: user2024@example.com
Username group    : user2024
Domain group      : example.com
Price matched     : 45.99


In [4]:
# Activity 1 (more on classes, quantifiers and groups)  |  Eman Amjad  |  FA23-BAI-013
# Character class + bounded quantifier: words of 5 or more word characters
print("Words with 5+ chars  :", re.findall(r"\b\w{5,}\b", sample_sentence))

# Custom character class: every run of digits (with an optional decimal part)
print("Numbers              :", re.findall(r"\d+(?:\.\d+)?", sample_sentence))

# With two groups, findall returns tuples instead of whole strings
emails = "user2024@example.com, a_b@test.org"
print("findall with groups  :", re.findall(email_pattern, emails))
print("findall, no groups   :", re.findall(r"\w+@\w+\.\w+", emails))

# Pitfall: \w+\.\w+ allows exactly one dot in the domain, so a multi-level domain is cut short
multi_level = "write to user@mail.example.co.uk"
print("Multi-level domain   :", re.search(email_pattern, multi_level).group(0))

Words with 5+ chars  : ['price', 'contact', 'email', 'user2024', 'example']
Numbers              : ['45.99', '2024']
findall with groups  : [('user2024', 'example.com'), ('a_b', 'test.org')]
findall, no groups   : ['user2024@example.com', 'a_b@test.org']
Multi-level domain   : user@mail.example


The groups split one match into a username (`user2024`) and a domain (`example.com`), and the price group returns only `45.99` without the `$`. When the pattern has groups, `findall` gives tuples of those groups; without groups it gives whole strings. The simple email pattern also shows its limit: on `user@mail.example.co.uk` it stops at `user@mail.example`, because the domain part only allows one dot.

**Activity 2** — Extract order numbers, emails and phone numbers from a customer message using `re.findall()`.

In [5]:
# Activity 2  |  Eman Amjad  |  FA23-BAI-013
customer_message = """
Hello, I am writing to inquire about my recent orders.
My first order number is ORD-58392 and the second one is ORD-99201.
You can reach me at john.doe@email.com or contact my alternate email jane_doe123@work.org.
My phone numbers are 555-123-4567 and (800) 555-9999.
"""

order_pattern = r"ORD-\d+"
# Wider class than \w so dots, underscores and plus signs are allowed in the local part
email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
# 3-3-4 digit groups, optional brackets around the area code, optional separator
phone_pattern = r"\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}"

extracted_orders = re.findall(order_pattern, customer_message)
extracted_emails = re.findall(email_pattern, customer_message)
extracted_phones = re.findall(phone_pattern, customer_message)

print("Extracted Orders:", extracted_orders)
print("Extracted Emails:", extracted_emails)
print("Extracted Phones:", extracted_phones)

Extracted Orders: ['ORD-58392', 'ORD-99201']
Extracted Emails: ['john.doe@email.com', 'jane_doe123@work.org']
Extracted Phones: ['555-123-4567', '(800) 555-9999']


In [6]:
# Activity 2 (phone pattern edge case)  |  Eman Amjad  |  FA23-BAI-013
# \(? and \)? are independent, so an unclosed bracket is still accepted
broken = "Call (800 555-9999 or 800) 555-1234"
print("Loose pattern  :", re.findall(phone_pattern, broken))

# Pairing the brackets with alternation rejects the malformed versions
strict_phone = r"(?:\(\d{3}\)\s?|\b\d{3}[-.\s]?)\d{3}[-.\s]?\d{4}\b"
print("Strict pattern :", re.findall(strict_phone, broken))
print("Strict on msg  :", re.findall(strict_phone, customer_message))

Loose pattern  : ['(800 555-9999', '800) 555-1234']
Strict pattern : ['800 555-9999']
Strict on msg  : ['555-123-4567', '(800) 555-9999']


Both order numbers, both emails (including `jane_doe123`, which needs the `_` inside the custom class) and both phone numbers are found. The loose phone pattern also accepts `(800 555-9999` and `800) 555-1234`, which have only one bracket each. The stricter version, which pairs the brackets, rejects `800) 555-1234`, keeps only the digits of `(800 555-9999` without the stray bracket, and still finds both valid numbers in the message.

**Activity 3** — Apply regex to a messier business-report excerpt to extract dates, percentages and revenue figures.

In [7]:
# Activity 3  |  Eman Amjad  |  FA23-BAI-013
business_report = """
In Q3, which ended on 2023-09-30, our overall growth was phenomenal.
We saw a 15.5% increase in user engagement and an 8% drop in churn rate compared to 12/31/2022.
Total revenue reached $1.2M in North America and $850,000 in Europe.
"""

# YYYY-MM-DD or MM/DD/YYYY
date_pattern = r"\d{4}-\d{2}-\d{2}|\d{1,2}/\d{1,2}/\d{4}"
# Integer or decimal followed by %
percentage_pattern = r"\d+(?:\.\d+)?%"
# $ amount with optional thousands commas, decimals and a K/M suffix
revenue_pattern = r"\$[\d,]+(?:\.\d+)?[KM]?"

extracted_dates = re.findall(date_pattern, business_report)
extracted_percentages = re.findall(percentage_pattern, business_report)
extracted_revenues = re.findall(revenue_pattern, business_report)

print("Extracted Dates      :", extracted_dates)
print("Extracted Percentages:", extracted_percentages)
print("Extracted Revenues   :", extracted_revenues)

Extracted Dates      : ['2023-09-30', '12/31/2022']
Extracted Percentages: ['15.5%', '8%']
Extracted Revenues   : ['$1.2M', '$850,000']


In [8]:
# Activity 3 (where the patterns break)  |  Eman Amjad  |  FA23-BAI-013
extra = "The audit closed on September 30, 2023. Costs rose by $3,400, then fell to $, 120."

# A third date format is outside the pattern's language, so it is silently missed
print("Dates found          :", re.findall(date_pattern, extra))
# [\d,]+ also accepts a lone comma, so '$,' is pulled in as a 'revenue'
print("Revenues found       :", re.findall(revenue_pattern, extra))

# Tighter versions: add a month-name date and require a digit right after $
month_date = r"(?:January|February|March|April|May|June|July|August|September|October|November|December) \d{1,2}, \d{4}"
better_date = date_pattern + "|" + month_date
better_revenue = r"\$\d[\d,]*(?:\.\d+)?[KM]?\b"
print("Better dates         :", re.findall(better_date, extra))
print("Better revenues      :", re.findall(better_revenue, extra))
print("Better on report     :", re.findall(better_revenue, business_report))

Dates found          :

 []
Revenues found       : ['$3,400,', '$,']
Better dates         : ['September 30, 2023']
Better revenues      : ['$3,400']
Better on report     : ['$1.2M', '$850,000']


On the report the three patterns work: 2 dates, 2 percentages (`15.5%`, `8%`) and 2 revenue figures (`$1.2M`, `$850,000`). On new text they are brittle: "September 30, 2023" is not found at all (no error, just an empty list), and `$,` is accepted as a revenue figure and `$3,400,` keeps the trailing comma. Adding a month-name alternative and requiring a digit right after `$` fixes both cases, but each new format still has to be added by hand.

**Activity 4** — Tokenize a sentence using both NLTK and spaCy and compare the two outputs.

In [9]:
# Activity 4  |  Eman Amjad  |  FA23-BAI-013
target_sentence = "Let's test this sentence! It costs $20.00."

nltk_tokens = word_tokenize(target_sentence)
spacy_tokens = [token.text for token in nlp(target_sentence)]
whitespace_tokens = target_sentence.split()   # naive baseline

print("Whitespace split:", whitespace_tokens)
print("NLTK Tokens     :", nltk_tokens)
print("spaCy Tokens    :", spacy_tokens)

# Exact list equality, not a visual check
print("\nNLTK == spaCy   :", nltk_tokens == spacy_tokens)
print(f"Token counts    : whitespace={len(whitespace_tokens)}, NLTK={len(nltk_tokens)}, spaCy={len(spacy_tokens)}")

Whitespace split: ["Let's", 'test', 'this', 'sentence!', 'It', 'costs', '$20.00.']
NLTK Tokens     : ['Let', "'s", 'test', 'this', 'sentence', '!', 'It', 'costs', '$', '20.00', '.']
spaCy Tokens    : ['Let', "'s", 'test', 'this', 'sentence', '!', 'It', 'costs', '$', '20.00', '.']

NLTK == spaCy   : True
Token counts    : whitespace=7, NLTK=11, spaCy=11


On this sentence NLTK and spaCy give exactly the same 11 tokens: `Let` + `'s` for the contraction, `!` and the final `.` as separate tokens, and `$` split from `20.00`. A plain `split()` gives only 7 tokens and leaves `sentence!` and `$20.00.` glued together. The two libraries do not always agree, which Home Activity 3 shows on a longer document.

**Activity 5** — Compare a blank spaCy pipeline (`spacy.blank("en")`) with a trained pipeline and explain why the blank pipeline only tokenizes.

In [10]:
# Activity 5  |  Eman Amjad  |  FA23-BAI-013
# Blank pipeline: vocabulary + rule-based tokenizer, no statistical components
blank_nlp = spacy.blank("en")
# Trained pipeline: same tokenizer plus tagger, parser, NER, ...
trained_nlp = nlp

print("Blank pipeline components  :", blank_nlp.pipe_names)
print("Trained pipeline components:", trained_nlp.pipe_names)

sample_text = "Apple is looking at buying a U.K. startup for $1 billion."
blank_document = blank_nlp(sample_text)
trained_document = trained_nlp(sample_text)

print("\n--- Blank Pipeline Output ---")
for token in blank_document:
    print(f"Token: {token.text:10} POS: {token.pos_:5} Entity: {token.ent_type_}")

print("\n--- Trained Pipeline Output ---")
for token in trained_document:
    print(f"Token: {token.text:10} POS: {token.pos_:5} Entity: {token.ent_type_}")

same_tokens = [t.text for t in blank_document] == [t.text for t in trained_document]
print("\nSame tokens in both pipelines:", same_tokens)
print("Tokens with a POS tag  (blank / trained):",
      sum(bool(t.pos_) for t in blank_document), "/", sum(bool(t.pos_) for t in trained_document))
print("Entities found         (blank / trained):",
      len(blank_document.ents), "/", [(e.text, e.label_) for e in trained_document.ents])

Blank pipeline components  : []
Trained pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

--- Blank Pipeline Output ---
Token: Apple      POS:       Entity: 
Token: is         POS:       Entity: 
Token: looking    POS:       Entity: 
Token: at         POS:       Entity: 
Token: buying     POS:       Entity: 
Token: a          POS:       Entity: 
Token: U.K.       POS:       Entity: 
Token: startup    POS:       Entity: 
Token: for        POS:       Entity: 
Token: $          POS:       Entity: 
Token: 1          POS:       Entity: 
Token: billion    POS:       Entity: 
Token: .          POS:       Entity: 

--- Trained Pipeline Output ---
Token: Apple      POS: PROPN Entity: ORG
Token: is         POS: AUX   Entity: 
Token: looking    POS: VERB  Entity: 
Token: at         POS: ADP   Entity: 
Token: buying     POS: VERB  Entity: 
Token: a          POS: DET   Entity: 
Token: U.K.       POS: PROPN Entity: GPE
Token: startup    POS: NOUN  Entity: 

Both pipelines produce the same 13 tokens (`U.K.` is kept whole and `$` is split from `1`), so tokenization works without any trained weights. The blank pipeline has an empty component list, so 0 of 13 tokens get a POS tag and no entities are found. The trained pipeline tags all 13 tokens and finds `Apple` (ORG), `U.K.` (GPE) and `$1 billion` (MONEY). The blank pipeline only tokenizes because spaCy's tokenizer is rule- and exception-based and lives in the `Language` object itself, while POS tags and entity labels come from statistical components (`tagger`, `parser`, `ner`) that only `spacy.load("en_core_web_sm")` adds. A blank pipeline is therefore the lighter choice when only segmentation is needed.

**Activity 6** — Filter tokens using token attributes (`is_alpha`, `like_num`, `like_email`, `is_currency`) instead of regex.

In [11]:
# Activity 6  |  Eman Amjad  |  FA23-BAI-013
unfiltered_text = "Contact me at alice@email.com. The total fee is $500, which includes 2 extra services."
document = nlp(unfiltered_text)

alphabetic_tokens, number_tokens, email_tokens, currency_tokens = [], [], [], []

# A token can land in more than one list, so these are separate ifs, not elif
for token in document:
    if token.is_alpha:
        alphabetic_tokens.append(token.text)
    if token.like_num:
        number_tokens.append(token.text)
    if token.like_email:
        email_tokens.append(token.text)
    if token.is_currency:
        currency_tokens.append(token.text)

print("Alphabetic Tokens:", alphabetic_tokens)
print("Number Tokens    :", number_tokens)
print("Email Tokens     :", email_tokens)
print("Currency Symbols :", currency_tokens)

# like_num also recognises number words, which a \d-based regex cannot
print("\nlike_num on words:", [(t.text, t.like_num) for t in nlp("two thousand and 3.5 or ten")])

Alphabetic Tokens:

 ['Contact', 'me', 'at', 'The', 'total', 'fee', 'is', 'which', 'includes', 'extra', 'services']
Number Tokens    : ['500', '2']
Email Tokens     : ['alice@email.com']
Currency Symbols : ['$']

like_num on words: [('two', True), ('thousand', True), ('and', False), ('3.5', True), ('or', False), ('ten', True)]


The attributes find the email `alice@email.com`, the numbers `500` and `2`, and 11 alphabetic words, without writing a single pattern. `$500` is not one unit: `is_currency` only flags the `$` symbol and `500` goes to the number list, so an amount has to be rebuilt from neighbouring tokens. `like_num` also recognises `two`, `thousand` and `ten` as numbers, which a digit regex would miss.

**Activity 7** — Perform sentence segmentation on a paragraph using both the NLTK and spaCy sentencizers.

In [12]:
# Activity 7  |  Eman Amjad  |  FA23-BAI-013
sample_paragraph = ("Natural Language Processing is fascinating. It allows computers to understand text. "
                    "Are you learning NLP right now? I hope so!")

nltk_sentences = sent_tokenize(sample_paragraph)                  # Punkt model
spacy_sentences = [s.text for s in nlp(sample_paragraph).sents]    # dependency parse

print("--- NLTK Sentences ---")
for index, sentence in enumerate(nltk_sentences, 1):
    print(f"{index}. {sentence}")

print("\n--- spaCy Sentences ---")
for index, sentence in enumerate(spacy_sentences, 1):
    print(f"{index}. {sentence}")

print("\nIdentical output:", nltk_sentences == spacy_sentences)

--- NLTK Sentences ---
1. Natural Language Processing is fascinating.
2. It allows computers to understand text.
3. Are you learning NLP right now?
4. I hope so!

--- spaCy Sentences ---
1. Natural Language Processing is fascinating.
2. It allows computers to understand text.
3. Are you learning NLP right now?
4. I hope so!

Identical output: True


Both give the same 4 sentences. This paragraph has no abbreviations or decimals, so every `.`, `?` and `!` really is a sentence end and the two methods cannot disagree. Home Activity 4 uses a harder paragraph where they do.

---
# Home Activities

**Home Activity 1** — Extract all emails, order numbers and phone numbers from the given raw text block using regex.

In [13]:
# Home Activity 1  |  Eman Amjad  |  FA23-BAI-013
raw_text = (DATA_DIR / "support_tickets.txt").read_text(encoding="utf-8")
print(raw_text)

email_regex = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
order_regex = r"\bORD-\d{5}\b"
phone_regex = r"(?:\(\d{3}\)\s?|\b\d{3}[-.\s]?)\d{3}[-.\s]?\d{4}\b"   # brackets paired

regex_results = {
    "emails": re.findall(email_regex, raw_text),
    "orders": re.findall(order_regex, raw_text),
    "phones": re.findall(phone_regex, raw_text),
}
for field, values in regex_results.items():
    print(f"{field:7} ({len(values)}): {values}")

# Groups turn each email into structured fields
print("\nEmail parts (user, domain):")
for user, domain in re.findall(r"([a-zA-Z0-9._%+-]+)@([a-zA-Z0-9.-]+\.[a-zA-Z]{2,})", raw_text):
    print(f"  {user:20} {domain}")

[Ticket #4471 | 2024-03-12 09:14]
From: sara.khan+orders@mail.shopnow.co.uk
Hi team, my order ORD-58392 arrived with only 3 of the 5 mugs I paid for ($48.50 total).
Please refund $19.40 or resend two mugs. You can call me at (415) 555-0198 after 6 pm.

[Ticket #4472 | 2024-03-12 10:02]
From: m.ali@techmart.pk
Orders ORD-61007 and ORD-61008 were both charged twice: 2 x $1,299.00 for the laptops.
I expect a refund of $2,598 within 7 days. Reach me at 212.555.7731 or billing@techmart.pk.

[Ticket #4473 | 2024-03-13 16:45]
From: j_doe99@gmail.com
Hello, I ordered twelve notebooks (ORD-70215) for €36 but received ten.
My backup number is 646 555 0142. Total spent this year is about £1,250, roughly 1500 dollars.

emails  (4): ['sara.khan+orders@mail.shopnow.co.uk', 'm.ali@techmart.pk', 'billing@techmart.pk', 'j_doe99@gmail.com']
orders  (4): ['ORD-58392', 'ORD-61007', 'ORD-61008', 'ORD-70215']
phones  (3): ['(415) 555-0198', '212.555.7731', '646 555 0142']

Email parts (user, domain):
  sara

The text block (three support tickets) gives 4 emails, 4 order numbers and 3 phone numbers, in three different phone formats (`(415) 555-0198`, `212.555.7731`, `646 555 0142`). The permissive email class handles `sara.khan+orders@mail.shopnow.co.uk`, which has a `+` tag and a multi-level domain; the Activity 1 pattern would have cut it short. The `\b` around the order and phone patterns stops them from matching inside ticket numbers or timestamps.

**Home Activity 2** — Extract all currency amounts and numeric quantities from the same text block using token attributes (not regex), and compare with the regex approach.

In [14]:
# Home Activity 2 (token attributes)  |  Eman Amjad  |  FA23-BAI-013
ticket_doc = nlp(raw_text)

# is_currency flags only the symbol, so an amount = currency token + the like_num token after it
token_amounts = []
for token in ticket_doc:
    if token.is_currency and token.i + 1 < len(ticket_doc) and ticket_doc[token.i + 1].like_num:
        token_amounts.append(token.text + ticket_doc[token.i + 1].text)

# Numeric quantities: like_num tokens that are not already part of a money amount
money_numbers = {t.i + 1 for t in ticket_doc if t.is_currency}
token_numbers = [t.text for t in ticket_doc if t.like_num and t.i not in money_numbers]

print("Currency amounts (tokens):", token_amounts)
print("Numeric tokens   (tokens):", token_numbers)

Currency amounts (tokens):

 ['$48.50', '$19.40', '$1,299.00', '$2,598', '€36', '£1,250']
Numeric tokens   (tokens): ['4471', '2024', '03', '12', '3', '5', 'two', '415', '555', '0198', '6', '4472', '2024', '03', '12', '2', '7', '212.555.7731', '4473', '2024', '03', '13', 'twelve', 'ten', '646', '555', '0142', '1500']


In [15]:
# Home Activity 2 (regex baseline and comparison)  |  Eman Amjad  |  FA23-BAI-013
regex_amounts = re.findall(r"[$€£]\d{1,3}(?:,\d{3})*(?:\.\d+)?", raw_text)
regex_numbers = re.findall(r"(?<![$€£\d,.])\b\d+(?:,\d{3})*(?:\.\d+)?\b", raw_text)

print("Currency amounts (regex):", regex_amounts)
print("Numbers          (regex):", regex_numbers)

comparison = pd.DataFrame({
    "method": ["token attributes", "regex"],
    "currency amounts": [len(token_amounts), len(regex_amounts)],
    "numeric items": [len(token_numbers), len(regex_numbers)],
})
print("\n" + comparison.to_string(index=False))

print("\nSame currency amounts      :", token_amounts == regex_amounts)
print("Numbers only tokens found  :", sorted(set(token_numbers) - set(regex_numbers)))
print("Numbers only regex found   :", sorted(set(regex_numbers) - set(token_numbers)))

# Neither symbol-based method sees '1500 dollars'; the trained NER does
print("\nNER MONEY entities:", [e.text for e in ticket_doc.ents if e.label_ == "MONEY"])

Currency amounts (regex): ['$48.50', '$19.40', '$1,299.00', '$2,598', '€36', '£1,250']
Numbers          (regex): ['4471', '2024', '03', '12', '09', '14', '58392', '3', '5', '415', '555', '0198', '6', '4472', '2024', '03', '12', '10', '02', '61007', '61008', '2', '7', '212.555', '4473', '2024', '03', '13', '16', '45', '70215', '646', '555', '0142', '1500']

          method  currency amounts  numeric items
token attributes                 6             28
           regex                 6             35

Same currency amounts      : True
Numbers only tokens found  : ['212.555.7731', 'ten', 'twelve', 'two']
Numbers only regex found   : ['02', '09', '10', '14', '16', '212.555', '45', '58392', '61007', '61008', '70215']

NER MONEY entities: ['48.50', '19.40', '1,299.00', '2,598', '36', 'about £1,250', 'roughly 1500 dollars']


For currency amounts the two methods agree exactly: both find the same 6 amounts (`$48.50` ... `£1,250`), but the token method needs the extra step of joining the `$` token with the number token after it, because `is_currency` never flags a whole amount. For quantities, `like_num` finds the number words `two`, `twelve` and `ten` that the digit regex cannot see, and it keeps `212.555.7731` as one token where the regex splits it. The regex returns 35 numeric items against 28 for the tokens: because `\b` treats `-` and `:` as boundaries, it also pulls the digits out of order IDs (`58392`, `61007`, ...) and timestamps (`09`, `14`, ...), which spaCy keeps inside single non-numeric tokens. Both methods still pick up digits that are not quantities (ticket numbers, date parts, phone pieces), so neither is clean without context. Neither catches `1500 dollars` as money, since there is no currency symbol; only the trained NER component labels it MONEY. Regex is more precise when the format is known; token attributes generalise better to wording the pattern did not expect.

**Home Activity 3** — Tokenize the given document using both NLTK and spaCy and note where the outputs differ (punctuation, contractions).

In [16]:
# Home Activity 3  |  Eman Amjad  |  FA23-BAI-013
document_text = (DATA_DIR / "product_review.txt").read_text(encoding="utf-8")
print(document_text, "\n")

nltk_doc_tokens = word_tokenize(document_text)
spacy_doc_tokens = [t.text for t in nlp(document_text)]
print(f"NLTK tokens : {len(nltk_doc_tokens)}")
print(f"spaCy tokens: {len(spacy_doc_tokens)}")
print("Identical   :", nltk_doc_tokens == spacy_doc_tokens)

# Align the two token lists and print only the places where they disagree
matcher = difflib.SequenceMatcher(a=nltk_doc_tokens, b=spacy_doc_tokens, autojunk=False)
rows = [{"NLTK": " | ".join(nltk_doc_tokens[i1:i2]), "spaCy": " | ".join(spacy_doc_tokens[j1:j2])}
        for op, i1, i2, j1, j2 in matcher.get_opcodes() if op != "equal"]
differences = pd.DataFrame(rows)
print(f"\n{len(differences)} differing spots:")
print(differences.to_string())

# Contractions: both libraries split the clitic off the same way
contractions = [t for t in nltk_doc_tokens if t.lower() in {"'ve", "'re", "n't", "'m", "'d", "'s"}]
print("\nClitics split by NLTK:", contractions)
print("Clitics split by spaCy:", [t for t in spacy_doc_tokens if t.lower() in {"'ve", "'re", "n't", "'m", "'d", "'s"}])

I've been using the X200 headphones for 3 weeks now, and honestly? They're great... mostly. The battery doesn't last the advertised 30 hrs; I'm getting ~22 hrs at 60% volume. Customer support (support@soundwave.com) said they'd "look into it" -- but it's been 5 days. Price: $129.99 (down from $149.99). Would I recommend 'em? Yes, if you can't find the Sony WH-1000XM4 on sale. P.S. The U.S. version ships with a USB-C cable, not a 3.5mm one! Rating: 4/5, see https://soundwave.com/x200-review. 

NLTK tokens : 119
spaCy tokens: 116
Identical   : False

8 differing spots:
                                      NLTK                              spaCy
0              support | @ | soundwave.com              support@soundwave.com
1                                       ``                                  "
2                                       ''                                  "
3                                   ' | em                                'em
4                                  P

NLTK gives 119 tokens and spaCy 116, with 8 differing spots:

- **Quotes:** NLTK rewrites straight double quotes into Penn Treebank form (` `` ` and `''`); spaCy keeps `"` as written.
- **Contractions:** the standard ones (`I've`, `They're`, `doesn't`, `I'm`, `they'd`, `it's`, `can't`) are split the same way by both. Only `'em` differs: NLTK splits it into `'` + `em`, spaCy keeps `'em` as one token.
- **Punctuation inside tokens:** NLTK breaks the email into `support | @ | soundwave.com` and the URL into `https | : | //soundwave...`, and splits `P.S.` into `P.S` + `.`. spaCy keeps the email, the URL and `P.S.` whole.
- **Hyphens and units:** spaCy splits `USB-C` into `USB | - | C` and `3.5mm` into `3.5 | mm`; NLTK keeps both whole.

spaCy's exceptions suit emails, URLs and abbreviations better, while NLTK keeps hyphenated product names together. Because the outputs differ, a pipeline should stick to one tokenizer.

**Home Activity 4** — Segment a 5–6 sentence paragraph using both the NLTK and spaCy sentencizers and flag any mismatch between them.

In [17]:
# Home Activity 4  |  Eman Amjad  |  FA23-BAI-013
paragraph = (DATA_DIR / "news_paragraph.txt").read_text(encoding="utf-8")
print(paragraph, "\n")

nltk_sents = sent_tokenize(paragraph)
spacy_sents = [s.text for s in nlp(paragraph).sents]


def end_offsets(sentences, text):
    """Character offset where each sentence ends, found by walking through the text."""
    ends, cursor = [], 0
    for sentence in sentences:
        start = text.index(sentence, cursor)
        cursor = start + len(sentence)
        ends.append(cursor)
    return ends


nltk_ends = set(end_offsets(nltk_sents, paragraph))
spacy_ends = set(end_offsets(spacy_sents, paragraph))

print(f"NLTK  : {len(nltk_sents)} sentences")
for i, s in enumerate(nltk_sents, 1):
    print(f"  {i}. {s}")
print(f"spaCy : {len(spacy_sents)} sentences")
for i, s in enumerate(spacy_sents, 1):
    print(f"  {i}. {s}")

print("\nIdentical output:", nltk_sents == spacy_sents)


def context(offset):
    return "..." + paragraph[max(0, offset - 25):offset] + " ||"


for offset in sorted(nltk_ends ^ spacy_ends):
    who = "spaCy only" if offset in spacy_ends else "NLTK only"
    print(f"MISMATCH ({who}) boundary after: {context(offset)}")
for offset in sorted(nltk_ends & spacy_ends):
    if paragraph[offset - 7:offset] == "approx.":
        print(f"SHARED ERROR (both split) after : {context(offset)}")

Last year, Dr. Ayesha Malik joined the U.S. office of DataCore Inc. She now leads a team of 12 engineers at a startup valued at $1.2 billion. Her first project, an NLP tool for customer e-mails, cut response times by 35 per cent. "We didn't expect results this fast," said J. R. Smith, the company's CTO. The tool now handles approx. 4,000 messages a day. Next quarter, the team plans to expand it to the U.K. and Canada.

NLTK  : 6 sentences
  1. Last year, Dr. Ayesha Malik joined the U.S. office of DataCore Inc. She now leads a team of 12 engineers at a startup valued at $1.2 billion.
  2. Her first project, an NLP tool for customer e-mails, cut response times by 35 per cent.
  3. "We didn't expect results this fast," said J. R. Smith, the company's CTO.
  4. The tool now handles approx.
  5. 4,000 messages a day.
  6. Next quarter, the team plans to expand it to the U.K. and Canada.
spaCy : 7 sentences
  1. Last year, Dr. Ayesha Malik joined the U.S. office of DataCore Inc.
  2. She now leads a team of 12 engineers at a startup valued at $1.2 billion.
  3. Her first project, an NLP tool for customer e-mails, cut response times by 35 per cent.
  4. "We didn't expect results this fast," said J. R. Smith, the company's CTO.
  5. The tool now handles approx.
  6. 4,000 messages a day.
  7. Next quarter, the team plans to expand it to the U.K. and Canada.

Identical output: False
MISMATCH (spaCy only) boun

The paragraph has 6 real sentences. NLTK returns 6 and spaCy 7, and the mismatch check flags one boundary: after "DataCore Inc." spaCy starts a new sentence and NLTK does not. Punkt has learned `Inc.` as an abbreviation, so it treats the period as non-final and merges the first two sentences; spaCy's parser sees that "She now leads ..." is a new clause and splits correctly. The decimal `$1.2`, `Dr.`, `U.S.`, `J. R.` and `U.K.` are handled correctly by both. The two methods also share one error: both split after "approx.", leaving "4,000 messages a day." as a fragment, so agreement between the tools does not prove that a boundary is correct.

---

### Lab 03 complete

7 in-lab activities and 4 home activities — submitted by Eman Amjad (FA23-BAI-013).